# 04 — Default Model: Cox Proportional Hazards

Spec §5.3. Data source: `panel_cph_2018_2025.parquet` (2018–2025 originations).

**Why CPH instead of logistic regression here:**  
Loans still performing at the observation end are right-censored — not confirmed
non-defaults. Logistic regression treats them as true zeros, biasing coefficients
downward. Cox PH conditions on the risk set at each event time, using censored
loans correctly up to their last observed month.

Pipeline:
1. Load 2018-2025 panel (never load the logistic panel here)
2. Aggregate to survival tuples with ELTV LOCF imputation
3. Random stratified train/test split
4. Fit CoxPHFitter(penalizer=0.1)
5. Print summary and check proportional hazards assumption
6. Compute C-index on test set
7. Plot baseline survival curve and partial hazard effects
8. Serialize model

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, str(Path('..') / 'src'))
import feature_engineering as fe
import survival_model as sm

PROCESSED = Path('..') / 'data' / 'processed'
FIGURES   = Path('..') / 'artifacts' / 'figures'
MODELS    = Path('..') / 'artifacts' / 'models'
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Load 2018-2025 panel

> **Critical:** Always load `panel_cph_2018_2025.parquet` here, never the logistic panel.
> The 2018-2020 cohorts provide the resolved terminal events (foreclosures) the
> survival model needs (spec §4.6).

In [ ]:
panel = pd.read_parquet(PROCESSED / 'panel_cph_2018_2025.parquet')
print(f'Panel shape: {panel.shape}')

# Verify 2018-2020 terminal events are present
panel['orig_year'] = pd.to_datetime(
    panel['first_payment_date'].astype(str), format='%Y%m', errors='coerce'
).dt.year
terminal_by_year = (
    panel[panel['zero_balance_code'] == '09']
    .groupby('orig_year').size()
)
print('REO/foreclosure events by origination year:')
print(terminal_by_year.to_string())
print('\nExpect non-zero counts in 2018-2020 (spec §10 Phase 2 check)')

## 2. Aggregate to survival tuples (ELTV LOCF, delinquency ordinal)

In [ ]:
survival_df = fe.build_survival_features(panel)
print(f'Survival dataset: {survival_df.shape}')
print(f'Event rate (ever 90+DPD): {survival_df.event.mean():.4%}')
print(f'Max duration: {survival_df.duration.max()} months')
print(f'Censoring rate: {(1-survival_df.event).mean():.1%}')
survival_df[['duration', 'event', 'fico_z', 'orig_cltv_z', 'delinq_last']].describe().round(3)

## 3. Random stratified train / test split

In [ ]:
train_df, valid_df, test_df = sm.temporal_split_survival(survival_df, train_frac=0.70, valid_frac=0.15)
for name, split in [('Train', train_df), ('Valid', valid_df), ('Test', test_df)]:
    print(f'{name:5s}: {len(split):>8,} loans | event rate {split.event.mean():.4%}')

## 4. Fit CoxPHFitter

In [ ]:
train_valid = pd.concat([train_df, valid_df])  # fit on combined train+valid
cph = sm.train_cox(train_valid, penalizer=0.1)
cph.print_summary()

## 5. Proportional hazards assumption check (Schoenfeld residuals)

Covariates with p < 0.05 violate the PH assumption — their effect is not constant
over loan age. Typical violations: FICO (strong early, weakens over time), ELTV
(strengthens as loan ages into negative equity).

In [ ]:
# check_assumptions prints plots and returns a results table
ph_results = sm.check_proportional_hazards(cph, train_valid, p_value_threshold=0.05)
print('\nNote: Covariates with p<0.05 violate PH assumption.')
print('Fix options: (1) include time-interaction term, (2) stratify baseline hazard')

## 6. C-index on test set

In [ ]:
c_idx = sm.cox_c_index(cph, test_df)
print(f'C-index (test set): {c_idx:.4f}')
print('Interpretation: fraction of admissible pairs correctly ranked by risk')
print('Expect > 0.70 for a well-specified mortgage default model')

## 7. Baseline survival curve and covariate effects

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline survival curve (median covariate values)
cph.baseline_survival_.plot(ax=axes[0])
axes[0].set_title('Baseline Survival Curve S₀(t)')
axes[0].set_xlabel('Loan Age (months)')
axes[0].set_ylabel('Survival Probability (no default)')
axes[0].set_ylim(0.9, 1.01)

# Partial hazard plot: vary FICO z-score, fix everything else at median
median_row = survival_df[sm.CPH_FEATURE_COLS].median()
fico_range = np.linspace(-3, 3, 7)
partial_hazards = []
for fico_z in fico_range:
    row = median_row.copy()
    row['fico_z'] = fico_z
    partial_hazards.append(float(cph.predict_partial_hazard(pd.DataFrame([row]))))

axes[1].plot(fico_range, partial_hazards, 'o-', color='steelblue')
axes[1].axhline(1.0, color='k', linestyle='--', alpha=0.4, label='Baseline hazard')
axes[1].set_title('Partial Hazard vs. FICO z-score\n(all other covariates at median)')
axes[1].set_xlabel('FICO z-score (higher = better credit)')
axes[1].set_ylabel('Relative Hazard')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / 'cox_survival_and_hazard.png', bbox_inches='tight')
plt.show()

In [ ]:
# Coefficient forest plot
fig, ax = plt.subplots(figsize=(8, 6))
cph.plot(ax=ax)
ax.set_title('Cox PH Coefficients (log hazard ratio ± 95% CI)\nPositive = higher default hazard')
plt.tight_layout()
plt.savefig(FIGURES / 'cox_coefficients.png', bbox_inches='tight')
plt.show()

## 8. Serialize model

In [ ]:
model_path = sm.save_cox_model(cph, out_dir=MODELS)
print(f'Cox model saved to: {model_path}')

# Verify round-trip
cph_loaded = sm.load_cox_model(model_path)
c_idx_reload = sm.cox_c_index(cph_loaded, test_df)
assert abs(c_idx - c_idx_reload) < 1e-6, 'C-index differs after reload!'
print(f'Round-trip load: C-index matches ({c_idx_reload:.4f}) ✓')